In [2]:
!pip install pymupdf requests pillow


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
import re
import requests
from io import BytesIO
import fitz  # PyMuPDF
from PIL import Image, UnidentifiedImageError
import json
import time

In [ ]:
OUT_IMAGES_DIR = "./images"
OUT_DATA_FILE = "./data.json"

In [5]:
RENDER_DPI = 300

# caption regex — looks for Figure / Fig / FIG / etc. (common patterns)
CAPTION_REGEX = re.compile(
    r'^\s*(Figure|Fig\.?|FIGURE)\s*(\d+)(\s*[:\.\-]|\s+\(|$)', re.IGNORECASE
)

In [6]:
DOWNLOAD_TIMEOUT = 30

# make output dirs
os.makedirs(OUT_IMAGES_DIR, exist_ok=True)

In [7]:
import requests
from io import BytesIO

def download_pdf_to_bytes(arxiv_id):
    """
    Download the PDF for an arXiv paper and return it as a BytesIO object.
    Returns None if the download fails or is not a valid PDF.
    """

    url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    }

    try:
        resp = requests.get(url, headers=headers, timeout=30)
    except Exception as e:
        print(f"Request failed for {arxiv_id}: {e}")
        return None

    if resp.status_code != 200:
        print(f"HTTP {resp.status_code} for {arxiv_id}")
        return None

    # Verify PDF magic bytes
    if not resp.content.startswith(b"%PDF"):
        print(f"Downloaded file for {arxiv_id} is not a PDF!")
        return None

    print(f"Downloaded PDF successfully for {arxiv_id}")
    return BytesIO(resp.content)


In [8]:
def overlap_x_fraction(rect1, rect2):
    """Calculates the horizontal overlap fraction between two rectangles."""
    x1 = max(rect1.x0, rect2.x0)
    x2 = min(rect1.x1, rect2.x1)
    if x2 <= x1:
        return 0.0

    width1 = rect1.width
    if width1 == 0:
         return 0.0

    return (x2 - x1) / width1


def bbox_distance_vertical(upper_rect, lower_rect):
    """distance in vertical axis from bottom of upper_rect to top of lower_rect (could be negative if overlap)"""
    return lower_rect.y0 - upper_rect.y1

In [ ]:
def process_pdf_bytes(arxiv_id, pdf_bytes_io):
    """
    Process a PDF to extract images, captions, and descriptive text.
    """
    arxiv_id = arxiv_id.replace(":", "-")
    doc = fitz.open(stream=pdf_bytes_io.read(), filetype="pdf")
    results = []

    # --- 1) Global Text Extraction & Indexing (Pre-pass) ---
    full_document_text = ""
    all_blocks_with_pos = []

    for pno in range(doc.page_count):
        page = doc[pno]
        blocks = page.get_text("blocks")
        for b in blocks:
            text_content = b[4].strip()
            if not text_content:
                continue

            start_idx = len(full_document_text)
            full_document_text += text_content + "\n"
            end_idx = len(full_document_text)

            all_blocks_with_pos.append({
                "text": text_content,
                "start": start_idx,
                "end": end_idx,
                "rect": fitz.Rect(b[:4]),
                "page": pno + 1
            })

    print(f"Index built. Total text length: {len(full_document_text)} chars.")

    # --- Main Processing Loop ---
    for pno in range(doc.page_count):
        page = doc[pno]
        page_num = pno + 1
        print(f"Processing Page {page_num}/{doc.page_count}")

        # 1) extract raster images
        images = []
        try:
            raw_images = page.get_images(full=True)
        except TypeError:
            raw_images = page.get_images()

        for imginfo in raw_images:
            xref = imginfo[0]
            try:
                img_bbox = page.get_image_bbox(xref)
                bbox = fitz.Rect(img_bbox)
            except Exception:
                bbox = None

            images.append({
                "xref": xref,
                "bbox": bbox,
                "base_img": doc.extract_image(xref),
            })

        # 2) extract text blocks for this page
        blocks = page.get_text("blocks")
        text_blocks = []
        for b in blocks:
            x0, y0, x1, y1, text, block_no = b[:6]
            # FIX 1: Replace newlines with spaces to handle multi-line captions cleaner
            clean_text = text.replace("\n", " ").strip()
            text_blocks.append({
                "rect": fitz.Rect(x0, y0, x1, y1),
                "text": clean_text,
            })
        text_blocks.sort(key=lambda tb: tb["rect"].y0)

        # 3) find caption blocks
        caption_blocks = []
        for idx, tb in enumerate(text_blocks):
            first_line = tb["text"].splitlines()[0] if tb["text"] else ""
            if CAPTION_REGEX.match(first_line):
                # FIX: Replace newlines with spaces
                clean_caption = tb["text"].replace("\n", " ").strip()

                caption_blocks.append({
                    "rect": tb["rect"],
                    "text": clean_caption,  # Use the cleaned text
                    "index": idx
                })

        # 4 & 5) Associate captions with images
        caption_assignments = []
        used_images = set()

        for cap in caption_blocks:
            cap_rect = cap["rect"]
            assigned = []

            for im_idx, im in enumerate(images):
                if im_idx in used_images: continue
                if im["bbox"] is None: continue

                im_bbox = im["bbox"]
                ox = overlap_x_fraction(im_bbox, cap_rect)

                if ox >= 0.15 and im_bbox.y1 <= cap_rect.y0 + (0.02 * page.rect.height):
                    assigned.append((im_idx, im))

            caption_assignments.append({
                "caption": cap["text"],
                "cap_rect": cap_rect,
                "images": assigned
            })
            for (im_idx, _) in assigned:
                used_images.add(im_idx)

        # Fallback: nearest caption for unused images
        for im_idx, im in enumerate(images):
            if im_idx in used_images: continue
            if im["bbox"] is None: continue

            best_cap_idx = None
            best_dist = None

            for ci, cap in enumerate(caption_assignments):
                dist = bbox_distance_vertical(im["bbox"], cap["cap_rect"])
                if best_dist is None or abs(dist) < abs(best_dist):
                    best_dist = dist
                    best_cap_idx = ci

            if best_cap_idx is not None and abs(best_dist) < (page.rect.height * 0.5):
                caption_assignments[best_cap_idx]["images"].append((im_idx, im))
                used_images.add(im_idx)

        # 6) Process assignments and EXTRACT DESCRIPTIONS
        fig_counter = 1
        for ca in caption_assignments:
            caption_text = ca["caption"]
            caption_rect = ca["cap_rect"]
            assigned_images = ca["images"]

            fig_match = CAPTION_REGEX.search(caption_text)
            fig_number = fig_match.group(2) if fig_match else None

            descriptions = []
            text_positions = []

            # Additional keys for the caption
            caption_out_text = caption_text
            caption_out_positions = []

            # --- Logic Start ---
            # Iterate over all text blocks in the document
            # FIX: Use index to allow lookahead for split descriptions
            num_blocks = len(all_blocks_with_pos)
            i = 0
            while i < num_blocks:
                block = all_blocks_with_pos[i]

                # Check if this block IS the caption itself
                is_caption_block = False
                # FIX: Explicit equality check
                block_clean = block['text'].replace('\n', ' ').strip()
                if block_clean == caption_text:
                    is_caption_block = True

                if block['page'] == page_num:
                     if block['rect'].intersects(ca['cap_rect']):
                          if abs(len(block['text']) - len(caption_text)) < 50:
                               is_caption_block = True

                if is_caption_block:
                    caption_out_positions.append([block['start'], block['end']])
                    i += 1
                    continue

                if fig_number:
                    ref_regex = re.compile(rf"(Figure|Fig\.?)\s*{re.escape(fig_number)}(?!\d)", re.IGNORECASE)
                    if ref_regex.search(block['text']):
                         desc_text = block['text']
                         # FIX: Handle split descriptions
                         stripped = desc_text.strip()
                         if not stripped.endswith(('.', '?', '!')):
                             if i + 1 < num_blocks:
                                 next_block = all_blocks_with_pos[i+1]
                                 desc_text += " " + next_block['text']

                         descriptions.append(desc_text)
                         text_positions.append([block['start'], block['end']])
                i += 1
            def add_result(img_path, img_ext, img_name_only):
                results.append({
                    "image_name": img_name_only,
                    "arxiv paper no.": arxiv_id,
                    "page number": page_num,
                    "figure number": fig_number,
                    "descriptions": descriptions,
                    "text positions": text_positions,
                    # NEW KEYS
                    "caption_text": caption_out_text,
                    "caption_positions": caption_out_positions
                })

            if assigned_images:
                # Standard matched images
                for img_local_index, (im_idx, im) in enumerate(assigned_images, start=1):
                    base_img = im["base_img"]
                    img_bytes = base_img["image"]
                    ext = base_img.get("ext", "png")

                    fname_fig_num = fig_number if fig_number else f"automatch_{fig_counter}"
                    base_name = f"{arxiv_id.replace('/','_')}_p{page_num}_fig{fname_fig_num}_img{img_local_index}.{ext}"
                    out_img_path = os.path.join(OUT_IMAGES_DIR, base_name)

                    with open(out_img_path, "wb") as f:
                        f.write(img_bytes)

                    add_result(out_img_path, ext, base_name)
            else:
                # Fallback Logic

                # --- FIX 2: Find Top Boundary (ignoring small labels) ---
                cap_y0 = ca["cap_rect"].y0
                # 1. Precise Bottom Boundary: STRICTLY top of caption (minus small padding)
                #    We do NOT want to capture any part of the caption text.
                crop_y1 = max(0, cap_y0 - 5)
                # 2. Smart Top Boundary Search
                #    Scan upwards for "body text". Ignore labels (short, no punctuation).
                found_top_limit = False
                prev_y = 0.0 # Default to top of page if nothing found
                current_tb_idx = -1
                for i, tb in enumerate(text_blocks):
                    if tb["rect"] == ca["cap_rect"]:
                        current_tb_idx = i
                        break
                if current_tb_idx > 0:
                     # Look backwards
                     for i in range(current_tb_idx - 1, -1, -1):
                         tb = text_blocks[i]
                         txt = tb["text"].strip()
                         # Check if this block is "body text" (stop condition)
                         is_long = len(txt) > 60
                         is_sentence = txt.endswith(('.', '?', '!'))
                         if is_long or is_sentence:
                             # FOUND BODY TEXT ABOVE => Stop here.
                             # The image top boundary starts AFTER this block.
                             prev_y = tb["rect"].y1
                             found_top_limit = True
                             break
                         # If it's not body text, we assume it's a label inside the image/graph area.
                         # We CONTINUE scanning upwards.
                # If we didn't find a hard text stop, limit height to ~400px default
                if not found_top_limit:
                     # Use the top of the page (0.0) or (caption - 400), whichever is closer to caption
                     # Actually, we want to go up to 400px, but not past 0.
                     prev_y = max(0.0, crop_y1 - 500)
                crop_y0 = max(0, prev_y)
                # 3. Minimum Constraints (Grow UPWARDS if needed)
                #    If the calculated height is too small (e.g. < 50px),
                #    we extend the TOP upwards, NOT the bottom.
                if (crop_y1 - crop_y0) < 50:
                     crop_y0 = max(0, crop_y1 - 250)
                # Define Crop Area
                # margin_w = min(0.15 * page.rect.width, 50)
                margin_w = 10
                crop_x0 = max(0, ca["cap_rect"].x0 - margin_w)
                crop_x1 = min(page.rect.width, ca["cap_rect"].x1 + margin_w)
                crop_rect = fitz.Rect(crop_x0, crop_y0, crop_x1, crop_y1)

                # --- FIX 3: Check for Embedded Images in Fallback Region ---
                found_embedded = False
                for im_idx, im in enumerate(images):
                    if im_idx in used_images: continue
                    if im["bbox"] is None: continue

                    intersect = im["bbox"] & crop_rect
                    if not intersect.is_empty:
                        im_area = im["bbox"].width * im["bbox"].height
                        int_area = intersect.width * intersect.height

                        # Match if intersection is > 50% of image
                        if im_area > 0 and (int_area / im_area) > 0.5:

                            # === INSERT CHECK HERE ===
                            # If image goes below the start of the caption, it might contain the caption text.
                            # In that case, we should skip this embedded image and let the
                            # rasterizer (below) handle the clean crop.
                            if im["bbox"].y1 > ca["cap_rect"].y0 + 5:
                                continue
                            # =========================
                            base_img = im["base_img"]
                            img_bytes = base_img["image"]
                            # ... (existing code to save and add result)
                            found_embedded = True

                # 4. Rasterize only if no embedded images found
                if not found_embedded:
                    zoom = 2.0
                    mat = fitz.Matrix(zoom, zoom)
                    try:
                        pix = page.get_pixmap(matrix=mat, clip=crop_rect, alpha=False)
                        fname_fig_num = fig_number if fig_number else f"fallback_{fig_counter}"
                        base_name = f"{arxiv_id.replace('/','_')}_p{page_num}_fig{fname_fig_num}_rasterized.png"
                        out_img_path = os.path.join(OUT_IMAGES_DIR, base_name)
                        pix.save(out_img_path)

                        add_result(out_img_path, "png", base_name)

                    except Exception as e:
                        print(f"Error rasterizing region: {e}")

            fig_counter += 1

    doc.close()
    return results

In [ ]:
ARXIV_IDS_FILE = "./paper_list_71.txt"

In [ ]:
def main():
    with open(ARXIV_IDS_FILE, "r", encoding="utf-8") as f:
        arxiv_ids = [line.strip() for line in f if line.strip()]
    files_count = 0
    images_count = 0
    all_results = []
    # with open(OUT_DATA_FILE, "r", encoding="utf-8") as f:
    #     all_results = json.load(f)
    for arxiv_id in arxiv_ids[357:]:
        print(f"\n=== Processing {arxiv_id} ===")
        pdf_io = download_pdf_to_bytes(arxiv_id)
        if not pdf_io:
            print(f" Could not download PDF for {arxiv_id}, skipping.")
            continue

        try:
            results = process_pdf_bytes(arxiv_id, pdf_io)
            all_results.extend(results)
            with open(OUT_DATA_FILE, "w", encoding="utf-8") as f:
                json.dump(all_results, f, indent=4, ensure_ascii=False)
            files_count += 1
            images_count += len(results)
            print(f" Finished {arxiv_id}: extracted {len(results)} items.")
            print(f"Total files processed: {files_count}, total images extracted: {images_count}")
        except Exception as e:
            print(f" Error processing {arxiv_id}: {e}")

In [12]:
main()


=== Processing arXiv:2403.04734 ===
Downloaded PDF successfully for arXiv:2403.04734
Index built. Total text length: 65139 chars.
Processing Page 1/15
Processing Page 2/15
Processing Page 3/15
Processing Page 4/15
Processing Page 5/15
Processing Page 6/15
Processing Page 7/15
Processing Page 8/15
Processing Page 9/15
Processing Page 10/15
Processing Page 11/15
Processing Page 12/15
Processing Page 13/15
Processing Page 14/15
Processing Page 15/15
 Finished arXiv:2403.04734: extracted 11 items.
Total files processed: 1, total images extracted: 11

=== Processing arXiv:2411.12191 ===
Downloaded PDF successfully for arXiv:2411.12191
Index built. Total text length: 35572 chars.
Processing Page 1/19
Processing Page 2/19
Processing Page 3/19
Processing Page 4/19
Processing Page 5/19
Processing Page 6/19
Processing Page 7/19
Processing Page 8/19
Processing Page 9/19
Processing Page 10/19
Processing Page 11/19
Processing Page 12/19
Processing Page 13/19
Processing Page 14/19
Processing Page 1

KeyboardInterrupt: 